In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.restaurantdb;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.restaurantdb.raw_data;

Before ingesting the data, you should upload the raw data csv files in the volume ( Catalog -> workspace -> restaurantdb -> raw_data then click on upload to volume and select the files ).


In [0]:
%sql
List '/Volumes/workspace/restaurantdb/raw_data'

In [0]:
def initialise_restaurantdb(tables_list, volume, schema) :

    """
    Ingests CSV files from a volume into Delta tables.

    Iterates through a list of file mappings, reads each CSV using spark, 
    and writes it as a Delta table in overwrite mode. Errors on individual 
    files are logged without stopping the overall execution.

    Args:
        tables_list (list of tuple): Mappings of (file_name, table_name).
        volume (str): Base path to the source CSV files.
        schema (str): Target database schema name.

    Returns:
        None

    Raises :
        Exception: If any error occurs during ingestion.
    """
    for file_name, table_name in tables_list:
        table_full_name = f"{schema}.{table_name}"
        file_path = f"{volume}/{file_name}"

        try :
            df = (
                spark.read
                .format("csv")
                .option("header", "true")
                .option("inferSchema", "true")
                .load(file_path)
            )
            df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_full_name)
            print(f"successfully ingested {table_full_name}")

        except Exception as e:
            print(f"Error ingesting {table_full_name}: {e}")

In [0]:
source_volume = "/Volumes/workspace/restaurantdb/raw_data"
source_db = "workspace.restaurantdb"
tables_to_ingest = [
    ("menu_raw.csv", "menu"),
    ("employees_raw.csv", "employees"),
    ("weather_raw.csv", "weather"),
    ("orders_raw.csv", "orders"),
    ("details_raw.csv", "order_details")]

initialise_restaurantdb(tables_to_ingest, source_volume, source_db)

In [0]:
%sql
SHOW TABLES IN workspace.restaurantdb;

In [0]:
tables = ["order_details", "menu", "employees", "orders", "weather"]
schema = "workspace.restaurantdb"

for table_name in tables:
    print(f"Schema for {schema}.{table_name}:")
    df = spark.table(f"{schema}.{table_name}")
    df.printSchema()
    print("\n")